# Step 4 — Build analysis-ready tables

This notebook prepares the cleaned CSV files for later analysis.

It will:

1. Combine the six detailed sector failure files vertically.
2. Keep the Crunchbase and synthetic-metrics datasets separate.
3. Keep the failure master file as a separate reference file.
4. Create a list of possible company matches for manual review.

**Important:** A matching normalized company name is not enough to prove that two
records describe the same company. The notebook therefore does not join the
failure and Crunchbase tables.


## Expected folder structure

Place this notebook in the project root, beside the `data` folder:

```text
project/
├── Step_4_Build_Analysis_Ready_Tables.ipynb
└── data/
    ├── cleaned/
    ├── processed/
    └── raw/
```

The notebook reads from `data/cleaned` and writes new files to
`data/processed`. It does not modify the cleaned files.


In [ ]:
# Import the two libraries needed in this notebook.
from pathlib import Path
import pandas as pd

# Define the project folders.
DATA_DIR = Path("data")
CLEANED_DIR = DATA_DIR / "cleaned"
PROCESSED_DIR = DATA_DIR / "processed"

if not CLEANED_DIR.exists():
    raise FileNotFoundError(
        "The data/cleaned folder was not found. "
        "Open this notebook from the project root."
    )

# Create the output folder if it does not already exist.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Cleaned data folder:", CLEANED_DIR)
print("Processed data folder:", PROCESSED_DIR)


## 1. Identify the input files

The six detailed failure files contain the failure flags needed for the
failure-pattern analysis. The master failure file is not included in this
vertical combination because it contains only name, sector, and operating years.


In [ ]:
# List the six detailed startup-failure files explicitly.
FAILURE_FILES = [
    "Startup Failure (Finance and Insurance)_clean.csv",
    "Startup Failure (Food and services)_clean.csv",
    "Startup Failure (Health Care)_clean.csv",
    "Startup Failure (Manufactures)_clean.csv",
    "Startup Failure (Retail Trade)_clean.csv",
    "Startup Failures (Information Sector)_clean.csv",
]

BIG_STARTUP_FILE = "big_startups_clean.csv"
METRICS_FILE = "synthetic_startup_metrics_clean.csv"
MASTER_FAILURE_FILE = "startup_failures_master_clean.csv"

# Stop early and show a clear message if an expected file is missing.
expected_files = FAILURE_FILES + [
    BIG_STARTUP_FILE,
    METRICS_FILE,
    MASTER_FAILURE_FILE,
]
missing_files = [
    name for name in expected_files
    if not (CLEANED_DIR / name).exists()
]

if missing_files:
    raise FileNotFoundError(f"Missing cleaned files: {missing_files}")

print("All expected cleaned files were found.")


## 2. Combine the six sector failure files vertically

Vertical combination means adding the rows of each sector file below the rows
of the previous file. Columns are aligned by column name.

Missing values in a failure flag remain missing. They are not changed to zero,
because a missing flag can mean that the factor was not collected in that file.


In [ ]:
failure_frames = []
file_checks = []

for file_name in FAILURE_FILES:
    table = pd.read_csv(CLEANED_DIR / file_name)

    # Record the source file before combining the rows.
    table["source_file"] = file_name
    failure_frames.append(table)

    file_checks.append(
        {
            "file": file_name,
            "rows": len(table),
            "columns": len(table.columns) - 1,
        }
    )

file_checks = pd.DataFrame(file_checks)
print(file_checks.to_string(index=False))

# Confirm that the cleaned files contain the same column names.
first_columns = set(failure_frames[0].columns) - {"source_file"}
same_schema = all(
    (set(table.columns) - {"source_file"}) == first_columns
    for table in failure_frames
)
print("\nAll six files have the same schema:", same_schema)


In [ ]:
# Add all failure records into one table.
failure_analysis = pd.concat(
    failure_frames,
    ignore_index=True,
    sort=False,
)

# Add a simple unique record ID for later analysis.
failure_analysis.insert(
    0,
    "failure_record_id",
    [f"FAIL_{number:04d}" for number in range(1, len(failure_analysis) + 1)],
)

# Repeated names are flagged, not deleted. A company can appear in more than one sector.
failure_analysis["name_appears_multiple_times"] = (
    failure_analysis["name_normalized"].duplicated(keep=False)
)

print("Combined failure-table shape:", failure_analysis.shape)
print("\nRecords by sector:")
print(failure_analysis["Sector"].value_counts().to_string())
print(
    "\nRecords whose normalized name appears more than once:",
    failure_analysis["name_appears_multiple_times"].sum(),
)


In [ ]:
# Save the combined failure table.
FAILURE_OUTPUT = PROCESSED_DIR / "failure_analysis_table.csv"
failure_analysis.to_csv(FAILURE_OUTPUT, index=False)

print("Saved:", FAILURE_OUTPUT)


## 3. Build the separate Crunchbase analysis table

Only columns relevant to funding, peer groups, geography, dates, and observed
status are retained. The table remains separate from the failure table.

`permalink` is used as the company identifier. Repeated company names are not
treated as duplicate companies.


In [ ]:
date_columns = ["founded_at", "first_funding_at", "last_funding_at"]
big_startups = pd.read_csv(
    CLEANED_DIR / BIG_STARTUP_FILE,
    parse_dates=date_columns,
    low_memory=False,
)

crunchbase_columns = [
    "permalink", "name", "name_normalized",
    "category_list", "primary_category",
    "funding_total_usd", "funding_total_missing", "funding_rounds",
    "status", "country_code", "state_code", "region", "city",
    "founded_at", "first_funding_at", "last_funding_at",
]
crunchbase_analysis = big_startups[crunchbase_columns].copy()

print("Crunchbase analysis-table shape:", crunchbase_analysis.shape)
print("Duplicate permalinks:", crunchbase_analysis["permalink"].duplicated().sum())
print("Missing funding totals:", crunchbase_analysis["funding_total_usd"].isna().sum())
print("\nObserved company statuses:")
print(crunchbase_analysis["status"].value_counts().to_string())


In [ ]:
# Save the separate Crunchbase table.
CRUNCHBASE_OUTPUT = PROCESSED_DIR / "crunchbase_analysis_table.csv"
crunchbase_analysis.to_csv(
    CRUNCHBASE_OUTPUT,
    index=False,
    date_format="%Y-%m-%d",
)

print("Saved:", CRUNCHBASE_OUTPUT)


## 4. Build the separate startup-metrics analysis table

The supplied and recalculated LTV/CAC and runway values were checked during
cleaning. This notebook keeps every row and adds one eligibility flag. A row is
eligible when both validation checks passed.


In [ ]:
metrics_analysis = pd.read_csv(CLEANED_DIR / METRICS_FILE)

# Convert validation values safely to True or False.
metrics_analysis["ltv_cac_valid"] = (
    metrics_analysis["ltv_cac_valid"].astype(str).str.lower().eq("true")
)
metrics_analysis["runway_valid"] = (
    metrics_analysis["runway_valid"].astype(str).str.lower().eq("true")
)

metrics_analysis["analysis_eligible"] = (
    metrics_analysis["ltv_cac_valid"]
    & metrics_analysis["runway_valid"]
)

print("Startup-metrics analysis-table shape:", metrics_analysis.shape)
print("Eligible records:", metrics_analysis["analysis_eligible"].sum())
print("Ineligible records:", (~metrics_analysis["analysis_eligible"]).sum())
print("\nHealth-status counts:")
print(metrics_analysis["health_status"].value_counts().to_string())


In [ ]:
# Save the separate startup-metrics table.
METRICS_OUTPUT = PROCESSED_DIR / "startup_metrics_analysis_table.csv"
metrics_analysis.to_csv(METRICS_OUTPUT, index=False)

print("Saved:", METRICS_OUTPUT)


## 5. Keep the master failure file separate

`startup_failures_master_clean.csv` contains broader failure coverage but does
not contain the detailed binary failure flags. It is therefore retained as a
reference table and is not appended to the six detailed files.


In [ ]:
failure_master = pd.read_csv(CLEANED_DIR / MASTER_FAILURE_FILE)

print("Master failure-reference shape:", failure_master.shape)
print("This table remains separate and is not included in failure_analysis_table.csv.")


## 6. Identify possible company matches without joining the datasets

The next cell finds exact matches on `name_normalized`. These are only candidates
for manual verification. Companies can share names, and the historical sources
may describe different entities or time periods.


In [ ]:
failure_names = failure_analysis.loc[
    failure_analysis["name_normalized"].notna(),
    ["failure_record_id", "Name", "Sector", "name_normalized"],
]

crunchbase_names = crunchbase_analysis.loc[
    crunchbase_analysis["name_normalized"].notna(),
    [
        "permalink", "name", "name_normalized", "status",
        "primary_category", "country_code",
    ],
]

potential_matches = failure_names.merge(
    crunchbase_names,
    on="name_normalized",
    how="inner",
)
potential_matches["verification_status"] = "Needs manual verification"

MATCH_OUTPUT = PROCESSED_DIR / "potential_company_matches.csv"
potential_matches.to_csv(MATCH_OUTPUT, index=False)

print("Candidate match rows:", len(potential_matches))
print("Unique normalized names:", potential_matches["name_normalized"].nunique())
print("Saved for review:", MATCH_OUTPUT)
print("No analysis table was joined using these candidates.")


## 7. Summarize the processed outputs

This summary records the tables created in Step 4. It is useful for checking
row counts before feature engineering begins.


In [ ]:
output_summary = pd.DataFrame(
    [
        {
            "table": FAILURE_OUTPUT.name,
            "rows": len(failure_analysis),
            "columns": len(failure_analysis.columns),
            "purpose": "Failure factors and sector comparisons",
        },
        {
            "table": CRUNCHBASE_OUTPUT.name,
            "rows": len(crunchbase_analysis),
            "columns": len(crunchbase_analysis.columns),
            "purpose": "Funding and observed-status comparisons",
        },
        {
            "table": METRICS_OUTPUT.name,
            "rows": len(metrics_analysis),
            "columns": len(metrics_analysis.columns),
            "purpose": "Runway and unit-economics analysis",
        },
        {
            "table": MATCH_OUTPUT.name,
            "rows": len(potential_matches),
            "columns": len(potential_matches.columns),
            "purpose": "Manual match review only",
        },
    ]
)

SUMMARY_OUTPUT = PROCESSED_DIR / "analysis_table_summary.csv"
output_summary.to_csv(SUMMARY_OUTPUT, index=False)

print(output_summary.to_string(index=False))
print("\nSaved summary:", SUMMARY_OUTPUT)


## Step 4 result

The three main datasets are now analysis-ready and remain separate:

- `failure_analysis_table.csv`
- `crunchbase_analysis_table.csv`
- `startup_metrics_analysis_table.csv`

`potential_company_matches.csv` is a review list only. It must not be used for a
cross-dataset analysis until the identities are manually verified.

Feature engineering is intentionally left for Step 5.
